# Create a Genie Space for Product Manual Comparison

This notebook creates a [Genie Space](https://docs.databricks.com/en/genie/) that lets users ask natural-language questions about **cordless drill/driver specifications** extracted from manufacturer PDF manuals.

The space is backed by the `app_productmanuals_processed` table, which contains structured specs (voltage, torque, speed, weight, etc.) for tools from Bosch, Makita, DeWalt, and Milwaukee.

**Example questions a user can ask:**
- "Which drill has the highest torque?"
- "Compare the weight of all 18V drills"
- "Which tool is compatible with ProCORE batteries?"

In [ ]:
# %pip install databricks-sdk -q

from databricks.sdk import WorkspaceClient
from manage_genie_space import *
w = WorkspaceClient()

TABLE_FULL_NAME = "data_extraction.default.app_productmanuals_processed"
WAREHOUSE_ID = "4b9b953939869799"

GENIE_SPACE_TITLE = "Power Tool Product Comparison"
DESCRIPTION = (
    "Compare cordless drill/driver specifications across manufacturers. "
    "Ask about voltage, torque, speed, weight, battery compatibility, and more. "
    f"Data source: {TABLE_FULL_NAME}"
)

SAMPLE_QUESTIONS = [
    "Which drill has the highest max torque?",
    "Compare all drills by weight and voltage",
    "List drills compatible with 18V batteries",
    "Which manufacturer offers the fastest high-speed RPM?",
    "Show me all specifications for the DeWalt DCD991",
]

SERIALIZED_SPACE = f"""{{
  "version": 2,
  "data_sources": {{
    "tables": [
      {{
        "identifier": "{TABLE_FULL_NAME}",
        "column_configs": [
          {{"column_name": "manufacturer", "enable_format_assistance": true, "enable_entity_matching": true}},
          {{"column_name": "model_number", "enable_format_assistance": true, "enable_entity_matching": true}},
          {{"column_name": "product_name", "enable_format_assistance": true}},
          {{"column_name": "product_type", "enable_format_assistance": true, "enable_entity_matching": true}},
          {{"column_name": "rated_voltage_v", "enable_format_assistance": true}},
          {{"column_name": "max_torque_nm", "enable_format_assistance": true}},
          {{"column_name": "no_load_speed_low_rpm", "enable_format_assistance": true}},
          {{"column_name": "no_load_speed_high_rpm", "enable_format_assistance": true}},
          {{"column_name": "weight_kg", "enable_format_assistance": true}},
          {{"column_name": "compatible_batteries", "enable_format_assistance": true}},
          {{"column_name": "compatible_chargers", "enable_format_assistance": true}}
        ]
      }}
    ]
  }},
  "sample_questions": {SAMPLE_QUESTIONS}
}}"""

In [ ]:
genie_space = get_existing_genie_space(w, GENIE_SPACE_TITLE)

In [ ]:
if not genie_space:
    response = create_genie_space(
        w, WAREHOUSE_ID, SERIALIZED_SPACE, GENIE_SPACE_TITLE, DESCRIPTION
    )
    print_space_info(response, created=True)
else:
    print_space_info(genie_space)